# ConvNeXt: horizontal-flip TTA from saved weights

No training, new weights, or new split. Restore the same `convnext_tiny.fb_in1k`
checkpoint and evaluation preprocessing as `cnn_2_inference.ipynb`.

**TTA (Test-Time Augmentation)** predicts the original image and a horizontal flip,
then averages their sigmoid probabilities: `p_tta = (p_original + p_flip) / 2`.
Threshold the averaged probabilities, not the logits or binary predictions.
Classification outputs have no spatial axes, so no output flip is needed.

1. Attach the competition data, the trained `best.pt`, and the **original** `folds.csv`.
   In the same Kaggle session they should already be under `/kaggle/working`.
2. Check the paths below. This notebook never creates or changes folds.
3. Run All. Compare baseline/TTA validation metrics before using the TTA submission.

Separate per-class thresholds are fitted on TTA validation probabilities. That fitted
F1 is a calibration result on the same fold, not an independent test score. The notebook
also reports both methods at 0.5 and at the saved checkpoint thresholds to expose the
effect of TTA independently of threshold fitting. Improvement is not guaranteed.

Outputs are written to `/kaggle/working/convnext_tta` and do not overwrite the earlier
`submission.csv`, checkpoint, or training results. Two forward passes per image make
inference approximately twice as expensive. W&B and its API key are not required.

In [1]:
# Run only if packages are missing, then restart the kernel before continuing.
# %pip install -q timm "albumentations>=2,<3"

## 1. Imports and paths

Set `CHECKPOINT_PATH` to the weights used for the 0.93519 submission.
`FOLDS_PATH = None` searches for the saved CSV; if multiple files are found, set it
explicitly to the one used in training. Keep zero workers for notebook compatibility.
Set `RUN_TEST_INFERENCE = False` if you only want the validation comparison first.

In [2]:
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display, FileLink
from sklearn.metrics import f1_score, precision_score, recall_score
from tqdm import tqdm
import torch
from torch.utils.data import Dataset, DataLoader
import timm

CHECKPOINT_PATH = Path('/kaggle/working/convnext_tiny_fb_in1k_fold0_resnet_pipeline/best.pt')
FOLDS_PATH = None
DATA_DIR = Path('/kaggle/input/competitions/cats-and-dogs')
OUTPUT_DIR = Path('/kaggle/working/convnext_tta')
BATCH_SIZE = 32
NUM_WORKERS = 0
RUN_TEST_INFERENCE = True
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

device: cuda


## 2. Restore the model and its preprocessing

`pretrained=False` here only avoids downloading initialization weights: every parameter
is restored from the trained checkpoint. The training model remains ImageNet-1k ConvNeXt.
Support both the current full-frame Albumentations preprocessing and legacy timm transforms.

In [3]:
if not CHECKPOINT_PATH.is_file():
    candidates = set()
    for search_root in [Path('/kaggle/working'), Path('/kaggle/input')]:
        for directory, subdirs, files in os.walk(search_root):
            subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
            if 'best.pt' in files:
                candidates.add(Path(directory) / 'best.pt')
    if len(candidates) != 1:
        raise FileNotFoundError(f'Set CHECKPOINT_PATH to the original saved weights. Candidates: {sorted(candidates)}')
    CHECKPOINT_PATH = candidates.pop()

checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=True)
config = checkpoint['config']
assert config['model_name'] == 'convnext_tiny.fb_in1k'
LABEL_COLS = list(config['label_cols'])
assert LABEL_COLS == ['Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
FOLD = int(config['fold'])
BASE_THRESHOLD = float(config.get('threshold', 0.5))
saved_thresholds = np.asarray(checkpoint.get('thresholds', [BASE_THRESHOLD] * len(LABEL_COLS)), dtype=np.float64)
assert 0 < BASE_THRESHOLD < 1
assert saved_thresholds.shape == (len(LABEL_COLS),) and np.isfinite(saved_thresholds).all()
assert ((saved_thresholds > 0) & (saved_thresholds < 1)).all()
model = timm.create_model(config['model_name'], pretrained=False, num_classes=len(LABEL_COLS))
model.load_state_dict(checkpoint['model'], strict=True)
model = model.to(DEVICE).eval()

preprocessing = config.get('preprocessing')
if preprocessing is None:
    eval_transform = timm.data.create_transform(**config['data_config'], is_training=False)
else:
    if preprocessing['name'] != 'albumentations_resize_v1':
        raise ValueError(f'Unknown preprocessing: {preprocessing}')
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    eval_transform = A.Compose([
        A.Resize(preprocessing['image_size'], preprocessing['image_size'],
                 interpolation=preprocessing['interpolation']),
        A.Normalize(mean=preprocessing['mean'], std=preprocessing['std']),
        ToTensorV2(),
    ])
print('Checkpoint:', CHECKPOINT_PATH, '| epoch:', checkpoint['epoch'], '| held-out fold:', FOLD)
print('Saved thresholds:', dict(zip(LABEL_COLS, saved_thresholds.tolist())))

Checkpoint: /kaggle/working/convnext_tiny_fb_in1k_fold0_resnet_pipeline/best.pt | epoch: 15 | held-out fold: 0
Saved thresholds: {'Cat': 0.5, 'Dog': 0.5, 'Moris': 0.5, 'Motya': 0.5, 'Biatrix': 0.5}


## 3. Reuse the original held-out fold

Read the original `folds.csv`; verify IDs, labels, fold assignments and group separation.
If training exported `validation_predictions.csv` next to this checkpoint, also verify
that the held-out image IDs match. Do not regenerate the split or use test labels.

In [4]:
if not ((DATA_DIR / 'train.csv').is_file() and (DATA_DIR / 'test.csv').is_file()):
    candidates = []
    for directory, subdirs, files in os.walk('/kaggle/input'):
        subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
        root = Path(directory)
        if {'train.csv', 'test.csv', 'sample_submission.csv'}.issubset(files):
            columns = pd.read_csv(root / 'train.csv', nrows=0).columns
            if {'image_id', *LABEL_COLS}.issubset(columns) and (root / 'train').is_dir() and (root / 'test').is_dir():
                candidates.append(root)
    if len(candidates) != 1:
        raise FileNotFoundError(f'Attach the competition data or set DATA_DIR. Candidates: {candidates}')
    DATA_DIR = candidates[0]

if FOLDS_PATH is None:
    saved_path = config.get('folds_path')
    local_paths = [Path('/kaggle/working/folds.csv'), Path('folds.csv'), CHECKPOINT_PATH.parent.parent / 'folds.csv']
    if saved_path:
        local_paths.append(Path(saved_path))
    candidates = {p.resolve() for p in local_paths if p.is_file()}
    if not candidates:
        for directory, subdirs, files in os.walk('/kaggle/input'):
            subdirs[:] = [n for n in subdirs if n not in {'train', 'test', 'images', 'wandb'}]
            if 'folds.csv' in files:
                candidates.add((Path(directory) / 'folds.csv').resolve())
    if len(candidates) != 1:
        raise FileNotFoundError(
            'Attach the ORIGINAL folds.csv from training or set FOLDS_PATH explicitly. '
            f'No new split will be created. Candidates: {sorted(candidates)}'
        )
    FOLDS_PATH = candidates.pop()
FOLDS_PATH = Path(FOLDS_PATH)

train = pd.read_csv(DATA_DIR / 'train.csv', dtype={'image_id': str})
folds = pd.read_csv(FOLDS_PATH, dtype={'image_id': str})
assert {'image_id', 'cluster', 'fold', *LABEL_COLS}.issubset(folds.columns)
assert train.image_id.notna().all() and train.image_id.is_unique
assert folds.image_id.notna().all() and folds.image_id.is_unique
assert set(train.image_id) == set(folds.image_id)
assert train[LABEL_COLS].isin([0, 1]).all().all()
aligned = folds.set_index('image_id').loc[train.image_id]
assert np.array_equal(train[LABEL_COLS].to_numpy(), aligned[LABEL_COLS].to_numpy()), 'Stale fold labels.'
assert folds[['cluster', 'fold']].notna().all().all()
assert set(folds.fold.unique()) == set(range(5))
assert folds.groupby('cluster').fold.nunique().eq(1).all(), 'A group crosses folds.'
frame = train.merge(folds[['image_id', 'cluster', 'fold']], on='image_id', validate='one_to_one', sort=False)
val_df = frame.loc[frame.fold == FOLD].reset_index(drop=True)
assert 0 < len(val_df) < len(frame)
assert set(val_df.cluster).isdisjoint(frame.loc[frame.fold != FOLD, 'cluster'])
previous_val_path = CHECKPOINT_PATH.parent / 'validation_predictions.csv'
if previous_val_path.is_file():
    previous_val = pd.read_csv(previous_val_path, dtype={'image_id': str})
    assert previous_val.image_id.is_unique and set(previous_val.image_id) == set(val_df.image_id), 'Different validation split.'
val_targets = val_df[LABEL_COLS].to_numpy(dtype=np.int32)
print('Data:', DATA_DIR, '| original folds:', FOLDS_PATH, '| validation images:', len(val_df))

Data: /kaggle/input/competitions/cats-and-dogs | original folds: /kaggle/working/folds.csv | validation images: 2448


## 4. Original and flipped predictions

The same deterministic transform is used for every image. Flip only the width axis
of the normalized NCHW tensor (`dims=[3]`), preserving channels and batch order.
Use two separate forward passes to keep peak memory close to ordinary inference.

In [5]:
class ImageRows(Dataset):
    def __init__(self, rows, split):
        self.ids = rows.image_id.tolist()
        self.split = split

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, index):
        image_path = DATA_DIR / self.split / f'{self.ids[index]}.jpg'
        with Image.open(image_path) as image:
            image = image.convert('RGB')
            if preprocessing is None:
                return eval_transform(image)
            return eval_transform(image=np.array(image))['image']

@torch.inference_mode()
def predict_pair(loader, description):
    model.eval()
    original_batches, tta_batches = [], []
    for images in tqdm(loader, desc=description):
        images = images.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            logits = model(images)
        original = logits.float().sigmoid()
        del logits
        with torch.autocast(device_type=DEVICE.type, enabled=DEVICE.type == 'cuda'):
            flipped_logits = model(torch.flip(images, dims=[3]))
        averaged = (original + flipped_logits.float().sigmoid()) * 0.5
        original_batches.append(original.cpu().numpy())
        tta_batches.append(averaged.cpu().numpy())
    original = np.concatenate(original_batches)
    tta = np.concatenate(tta_batches)
    for probabilities in (original, tta):
        assert probabilities.shape == (len(loader.dataset), len(LABEL_COLS))
        assert np.isfinite(probabilities).all() and ((probabilities >= 0) & (probabilities <= 1)).all()
    return original, tta

val_loader = DataLoader(ImageRows(val_df, 'train'), batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
val_original, val_tta = predict_pair(val_loader, 'validation original + flip')

validation original + flip: 100%|██████████| 77/77 [00:26<00:00,  2.88it/s]


## 5. Validation comparison and TTA thresholds

Compare at 0.5 and at the saved thresholds before fitting TTA thresholds. Reuse the
training notebook's threshold search (0.05 to 0.95, step 0.01, tie nearest 0.5).
Save both raw probability variants so the comparison can be inspected without another
GPU pass. A higher fitted validation F1 does not guarantee a higher Kaggle score.

In [6]:
def tune_thresholds(targets, probabilities):
    thresholds = np.full(len(LABEL_COLS), 0.5, dtype=np.float64)
    candidates = np.round(np.linspace(0.05, 0.95, 91), 2)
    for j, label in enumerate(LABEL_COLS):
        if np.unique(targets[:, j]).size < 2:
            print(f'{label}: keeping 0.5; validation lacks one outcome.')
            continue
        scores = np.array([f1_score(targets[:, j], probabilities[:, j] >= t, zero_division=0)
                           for t in candidates])
        tied = np.flatnonzero(np.isclose(scores, scores.max(), rtol=0, atol=1e-12))
        best = tied[np.argmin(np.abs(candidates[tied] - 0.5))]
        thresholds[j] = candidates[best]
    return thresholds

def scores(probabilities, thresholds):
    predictions = (probabilities >= thresholds).astype(np.int32)
    return {
        'macro_f1': float(f1_score(val_targets, predictions, average='macro', zero_division=0)),
        'exact_match_accuracy': float((predictions == val_targets).all(axis=1).mean()),
    }

tta_thresholds = tune_thresholds(val_targets, val_tta)
comparison = pd.DataFrame.from_dict({
    'original @ 0.5': scores(val_original, 0.5),
    'TTA @ 0.5': scores(val_tta, 0.5),
    'original @ saved thresholds': scores(val_original, saved_thresholds),
    'TTA @ saved thresholds': scores(val_tta, saved_thresholds),
    'TTA @ fitted TTA thresholds': scores(val_tta, tta_thresholds),
}, orient='index')
display(comparison)
print('Checkpoint fixed-threshold F1:', checkpoint['metrics'].get('macro_f1'))
print('Recomputed fixed-0.5 F1:', comparison.loc['original @ 0.5', 'macro_f1'])
baseline_f1 = comparison.loc['original @ saved thresholds', 'macro_f1']
tta_f1 = comparison.loc['TTA @ fitted TTA thresholds', 'macro_f1']
print(f'TTA fitted F1 minus saved-threshold baseline: {tta_f1 - baseline_f1:+.6f}')
if tta_f1 < baseline_f1:
    print('TTA did not improve fitted validation F1. Keep the baseline as a candidate; TTA is still exported separately.')

per_label_rows = []
for j, label in enumerate(LABEL_COLS):
    for method, probabilities, cutoffs in [('original', val_original, saved_thresholds), ('tta', val_tta, tta_thresholds)]:
        predicted = probabilities[:, j] >= cutoffs[j]
        per_label_rows.append({
            'label': label, 'method': method, 'positives': int(val_targets[:, j].sum()),
            'threshold': float(cutoffs[j]),
            'f1': float(f1_score(val_targets[:, j], predicted, zero_division=0)),
            'precision': float(precision_score(val_targets[:, j], predicted, zero_division=0)),
            'recall': float(recall_score(val_targets[:, j], predicted, zero_division=0)),
        })
per_label = pd.DataFrame(per_label_rows)
display(per_label)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(OUTPUT_DIR / 'validation_comparison.csv', index_label='method')
per_label.to_csv(OUTPUT_DIR / 'validation_per_label.csv', index=False)
validation_predictions = val_df[['image_id', 'cluster', 'fold', *LABEL_COLS]].copy()
for j, label in enumerate(LABEL_COLS):
    validation_predictions[f'{label}_prob_original'] = val_original[:, j]
    validation_predictions[f'{label}_prob_tta'] = val_tta[:, j]
validation_predictions.to_csv(OUTPUT_DIR / 'validation_probabilities.csv', index=False)
tta_config = {
    'checkpoint_path': str(CHECKPOINT_PATH), 'checkpoint_epoch': int(checkpoint['epoch']),
    'folds_path': str(FOLDS_PATH), 'fold': FOLD, 'label_cols': LABEL_COLS,
    'views': ['original', 'horizontal_flip'], 'aggregation': 'mean_sigmoid_probabilities',
    'thresholds': dict(zip(LABEL_COLS, tta_thresholds.tolist())),
    'baseline_saved_thresholds': dict(zip(LABEL_COLS, saved_thresholds.tolist())),
    'preprocessing': preprocessing, 'data_config': config['data_config'],
    'validation_f1_baseline': float(baseline_f1), 'validation_f1_tta_fitted': float(tta_f1),
    'threshold_source': 'same_held_out_fold_as_training_checkpoint',
}
(OUTPUT_DIR / 'tta_config.json').write_text(json.dumps(tta_config, indent=2))
print('TTA thresholds:', tta_config['thresholds'])

,macro_f1,exact_match_accuracy
original @ 0.5,0.972387,0.993056
TTA @ 0.5,0.972787,0.993464
original @ saved thresholds,0.972387,0.993056
TTA @ saved thresholds,0.972787,0.993464
TTA @ fitted TTA thresholds,0.972890,0.993873


Checkpoint fixed-threshold F1: 0.972387075747738
Recomputed fixed-0.5 F1: 0.972387075747738
TTA fitted F1 minus saved-threshold baseline: +0.000503


,label,method,positives,threshold,f1,precision,recall
0,Cat,original,1006,0.50,0.993536,0.994030,0.993042
1,Cat,tta,1006,0.50,0.995025,0.996016,0.994036
2,Dog,original,982,0.50,0.993890,0.993890,0.993890
3,Dog,tta,982,0.49,0.994914,0.993902,0.995927
4,Moris,original,14,0.50,1.000000,1.000000,1.000000
5,Moris,tta,14,0.50,1.000000,1.000000,1.000000
6,Motya,original,7,0.50,0.933333,0.875000,1.000000
7,Motya,tta,7,0.50,0.933333,0.875000,1.000000
8,Biatrix,original,9,0.50,0.941176,1.000000,0.888889
9,Biatrix,tta,9,0.50,0.941176,1.000000,0.888889


TTA thresholds: {'Cat': 0.5, 'Dog': 0.49, 'Moris': 0.5, 'Motya': 0.5, 'Biatrix': 0.5}


## 6. Test inference and separate submissions

Export `submission_tta.csv` with the new TTA probabilities and fitted TTA thresholds.
Also export `submission_baseline.csv` from the original predictions and saved thresholds
as a control. Align both by `image_id` to the official sample submission, never by
accidental CSV row order. The existing training checkpoint is read-only.

Download `convnext_tta/submission_tta.csv` from Kaggle Output to submit it. The Kaggle
score is the external comparison with the previous 0.93519 result; no submission is
uploaded automatically. Set `RUN_TEST_INFERENCE = False` above to skip this section.

In [7]:
if RUN_TEST_INFERENCE:
    test = pd.read_csv(DATA_DIR / 'test.csv', dtype={'image_id': str})
    assert len(test) > 0 and test.image_id.notna().all() and test.image_id.is_unique
    sample = pd.read_csv(DATA_DIR / 'sample_submission.csv', dtype={'image_id': str})
    expected_columns = ['image_id', 'Cat', 'Dog', 'Moris', 'Motya', 'Biatrix']
    assert set(sample.columns) == set(expected_columns)
    assert sample.image_id.notna().all() and sample.image_id.is_unique
    assert set(sample.image_id) == set(test.image_id)
    test_loader = DataLoader(ImageRows(test, 'test'), batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=DEVICE.type == 'cuda')
    test_original, test_tta = predict_pair(test_loader, 'test original + flip')
    for method, probabilities, cutoffs in [('baseline', test_original, saved_thresholds), ('tta', test_tta, tta_thresholds)]:
        probability_frame = pd.DataFrame(probabilities, columns=LABEL_COLS)
        probability_frame.insert(0, 'image_id', test.image_id.to_numpy())
        probability_frame.to_csv(OUTPUT_DIR / f'test_probabilities_{method}.csv', index=False)
        submission = probability_frame.copy()
        submission[LABEL_COLS] = (probabilities >= cutoffs).astype(np.int32)
        submission = sample[['image_id']].merge(submission, on='image_id', how='left', validate='one_to_one')
        submission = submission[expected_columns]
        assert len(submission) == len(test) and not submission.isna().any().any()
        assert submission[LABEL_COLS].isin([0, 1]).all().all()
        output_path = OUTPUT_DIR / f'submission_{method}.csv'
        submission.to_csv(output_path, index=False)
        print('Saved:', output_path, '| rows:', len(submission))
        display(FileLink(str(output_path)))
        if method == 'tta':
            display(submission.head())
else:
    print('Test inference skipped; inspect validation comparison above.')

test original + flip: 100%|██████████| 584/584 [02:15<00:00,  4.32it/s]


Saved: /kaggle/working/convnext_tta/submission_baseline.csv | rows: 18673


/kaggle/working/convnext_tta/submission_baseline.csv

Saved: /kaggle/working/convnext_tta/submission_tta.csv | rows: 18673


/kaggle/working/convnext_tta/submission_tta.csv

,image_id,Cat,Dog,Moris,Motya,Biatrix
0,12c79b0a4bd444ca7e5f730b86824b9279d3cd24,0,0,0,0,0
1,dd9440332972c9c5cecb5bf3cfb3329d6b2e422c,0,1,0,0,0
2,ac89f642faef21c92d622d882943e9065600354a,0,0,0,0,0
3,7245e7046ae54da95dbddc24ae814b3c61cc6237,1,0,0,0,0
4,d2b76179615de57b54cb00d094fba50810d96127,1,0,0,0,0
